# A2UI Jupyter smoke test

This notebook runs unchanged on CPython (JupyterLab), Pyodide, and XPython. The setup cell sets `KERNEL` to the detected kernel: `cpython`, `pyodide`, or `xpython`.

Run the setup cell and the surface cell, enter a name, and click **Submit**. The callback replaces the form with a confirmation. Run the last cell to inspect the action.

If a saved output reports that its widget model was not found, clear the output and rerun the surface cell. See the [README](../README.md#saved-widget-output) for saving widget state.

In [ ]:
import sys
from importlib.util import find_spec

# Pyodide and xeus-python both report sys.platform "emscripten"; only Pyodide ships the pyodide package.
if sys.platform != "emscripten":
    KERNEL = "cpython"
else:
    try:
        has_pyodide = find_spec("pyodide") is not None
    except ValueError:  # xeus-python's pyodide stub has __spec__ set to None
        has_pyodide = False
    KERNEL = "pyodide" if has_pyodide else "xpython"

if KERNEL == "pyodide" and find_spec("a2ui_jupyter") is None:
    import piplite

    await piplite.install(["rpds-py", "a2ui-jupyter"])

from a2ui_jupyter import A2UISurface

BASIC_CATALOG_ID = "https://a2ui.org/specification/v1_0/catalogs/basic/catalog.json"


def new_surface(surface_id, *, send_data_model=False, **widget_kwargs):
    surface = A2UISurface(**widget_kwargs)
    create = {"surfaceId": surface_id, "catalogId": BASIC_CATALOG_ID}
    if send_data_model:
        create["sendDataModel"] = True
    surface.send({"version": "1.0", "createSurface": create})
    return surface

In [ ]:
from IPython.display import display

surface = new_surface("manual-smoke-test")
surface.send(
    [
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "manual-smoke-test",
                "path": "/person/name",
                "value": "",
            },
        },
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "manual-smoke-test",
                "path": "/status",
                "value": "Waiting for submission",
            },
        },
        {
            "version": "1.0",
            "updateComponents": {
                "surfaceId": "manual-smoke-test",
                "components": [
                    {
                        "id": "root",
                        "component": "Column",
                        "children": ["title", "name", "submit", "status"],
                        "align": "stretch",
                    },
                    {
                        "id": "title",
                        "component": "Text",
                        "text": "A2UI Jupyter renderer smoke test",
                    },
                    {
                        "id": "name",
                        "component": "TextField",
                        "label": "Name",
                        "placeholder": "Enter a name",
                        "value": {"@path": "/person/name"},
                    },
                    {
                        "id": "submit",
                        "component": "Button",
                        "child": "submit_label",
                        "variant": "primary",
                        "action": {
                            "event": {
                                "name": "submit",
                                "context": {
                                    "name": {"@path": "/person/name"},
                                },
                            },
                        },
                    },
                    {
                        "id": "submit_label",
                        "component": "Text",
                        "text": "Submit",
                    },
                    {
                        "id": "status",
                        "component": "Text",
                        "text": {"@path": "/status"},
                    },
                ],
            },
        },
    ]
)


action_data = None


def handle_action(action):
    global action_data
    action_data = action.model_dump(by_alias=True, exclude_unset=True)
    # Replace the form with a confirmation so it cannot be submitted again.
    surface.send(
        [
            {
                "version": "1.0",
                "updateDataModel": {
                    "surfaceId": "manual-smoke-test",
                    "path": "/status",
                    "value": f"Submitted: {action_data['action']['context']['name']}",
                },
            },
            {
                "version": "1.0",
                "updateComponents": {
                    "surfaceId": "manual-smoke-test",
                    "components": [
                        {
                            "id": "root",
                            "component": "Column",
                            "children": ["title", "status"],
                            "align": "stretch",
                        },
                    ],
                },
            },
        ]
    )


surface.on_action(handle_action)
display(surface)

After submitting, inspect the action:

In [ ]:
action_data